# Solution: Gaussian Process Regression

In [ ]:
import torch
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
from cycler import cycler
import seaborn as sns

# Set the color scheme
sns.set_theme()
colors = [
    "#0076C2",
    "#EC6842",
    "#A50034",
    "#009B77",
    "#FFB81C",
    "#E03C31",
    "#6CC24A",
    "#EF60A3",
    "#0C2340",
    "#00B8C8",
    "#6F1D77",
]
plt.rcParams["axes.prop_cycle"] = cycler(color=colors)

seed = 1
torch.manual_seed(seed)
np.random.seed(seed)

# By default torch uses float32, this can lead to numerical problems for the (semi-)positive definiteness of tensors.
torch.set_default_dtype(torch.float64)

## Introduction
The goal of this exercise is to develop a deeper understanding of how kernel methods work.
We will code the general Gaussian Process framework and test it on a simple dataset.
Finally, to get a good model, we perform model selection.

Let's start by creating our dataset as usual:

In [ ]:
# The true function relating t to x
def f_truth(x):
    # Return a sine
    return torch.sin(2 * x)

# The data is generated from the ground truth with i.i.d. gaussian noise
def f_data(x, rng, epsilon=0.2):
    # Generate N noisy observations (1 at each location)
    t = f_truth(x) + torch.normal(mean=0, std=epsilon, size=x.shape, generator=rng)

    return t

## The kernel

As we have seen in the lecture, many different covariance functions can be constructed from a set of elementary kernels, such as the polynomial or the Matérn kernels.
The choice of this function has a large influence on the model's prediction.
In engineering applications, you often have some idea of what trends are present in the data, which can inform your choice for the covariance function.

### The squared exponential kernel

Let's start with the implementation of arguably the most popular choice of kernel, the so-called squared exponential kernel that you already know from the lecture:

$$
k(\mathbf{x},\mathbf{x}') = \sigma_f^2\exp\left(\displaystyle-\frac{1}{2\ell^2}\lVert\mathbf{x}-\mathbf{x}'\rVert^2\right).
$$

It has two hyperparameters with the scaling factor $\sigma_f$ and the length scale $\ell$ that controls the correlation between data points. We pass these parameters as `**hyperparams` to make it simple to switch between covariance functions with different parameters.

In [ ]:
def cov_func(X1, X2, **hyperparams):
    sig_f = hyperparams["sig_f"]
    length = hyperparams["length"]

    out = sig_f**2 * torch.exp(-torch.cdist(X1, X2) ** 2 / (2 * length**2))

    return out

See how we re-implemented the kernel from the previous notebook but now in a more general way that uses Torch and accepts inputs of any dimensionality.

We set initial values for the parameters and plot the result:

In [ ]:
hyperparams = {
    "sig_f": 1,
    "length": 1,
}

def plot_covariance(func):
    fig, ax1 = plt.subplots(1, 1, figsize=(4, 3))
    xlim = (-1, 1)
    X = np.expand_dims(np.linspace(*xlim, 200), 1)
    X = torch.tensor(X)
    # Note how we can also get the full covariance matrix without any for loops:
    cov = func(X, X, **hyperparams)

    # Plot covariance matrix
    im = ax1.imshow(cov)
    cbar = plt.colorbar(im, ax=ax1, fraction=0.045, pad=0.05)
    cbar.ax.set_ylabel("$k(x,x)$", fontsize=10)
    ax1.set_title(("Example of covariance matrix"))
    ax1.set_xlabel("x", fontsize=13)
    ax1.set_ylabel("x", fontsize=13)
    ax1.grid(False)


plot_covariance(cov_func)

## Sample realizations

In [ ]:
# Sample from the Gaussian process distribution
n_points = 100  # Number of data points in each function

n_samples = 10  # Number of GP samples
# Independent variable samples
X = torch.linspace(-2, 2, n_points).view(-1, 1)
cov = cov_func(X, X, **hyperparams)

# Draw samples from a multivariate normal function with our covariance function
samples = torch.tensor(
    np.random.multivariate_normal(mean=np.zeros(n_points), cov=cov, size=n_samples)
)

# Plot the sampled functions
fig, ax = plt.subplots(1, 1, figsize=(8, 3))
for i in range(n_samples):
    ax.plot(X, samples[i], linestyle="-", marker="o", markersize=3)
ax.set_xlabel("$x$", fontsize=13)
ax.set_ylabel("$y = f(x)$", fontsize=13)
ax.set_title(f"{n_samples} different GP realizations at {n_points} points\n")
ax.set_xlim([-2.1, 2.1])
plt.show()

## Exercise: Implementing the Gaussian Process
Now we are ready to implement a Gaussian Process (GP) for regression.

In the lecture, we have already seen how to compute the posterior predictive distribution given the data in a single prediction location $p(\hat{t}\vert\mathbf{t})$.
This can be generalized to a vector of multiple prediction locations $p(\hat{\mathbf{t}}\vert\mathbf{t})$. Furthermore, we can also predict function values $p(\hat{\mathbf{y}}\lvert\mathbf{t})$ instead of $\hat{\mathbf{t}}$ by removing the contribution of the noise from the posterior covariance.
In that case, the posterior predictive distribution is given by:

$$
p(\hat{\mathbf{y}}\vert\mathbf{t}) = \mathcal{N}\left(
\hat{\mathbf{y}}\vert \hat{\mathbf{m}},\hat{\mathbf{\Sigma}}
\right),
$$

with posterior mean

$$
\hat{\mathbf{m}} = K\left(\hat{\mathbf{X}},\mathbf{X}\right)
\left[
K\left(\mathbf{X},\mathbf{X}\right) + \beta^{-1}\mathbf{I}
\right]^{-1}
\mathbf{t},
$$

and posterior variance

$$
\hat{\mathbf{\Sigma}} = {K\left(\hat{\mathbf{X}},\hat{\mathbf{X}}\right)} - K\left(\hat{\mathbf{X}},\mathbf{X}\right)
\left[
K\left(\mathbf{X},\mathbf{X}\right) + \beta^{-1}\mathbf{I}
\right]^{-1}
{K\left(\mathbf{X},\hat{\mathbf{X}}\right)}.
$$

Complete the function below to compute the posterior mean and covariance given the observed data $(\mathbf{X}, \mathbf{t})$ and a set of prediction locations $\hat{\mathbf{X}}$.

In [ ]:
# Gaussian process posterior
def GP(X, t, X_hat, kernel, hyperparams):
    """
    :param X: Observation locations [N]
    :param t: Observation values [N]
    :param X_hat: Prediction locations [Np]
    :param kernel: covariance function
    :param hyperparams: The hyperparameters
    :return: posterior mean [Np] and covariance matrix [Np,Np]
    """
    with torch.no_grad():
        noise = hyperparams["noise"]

        # ---------------------- student exercise --------------------------------- #
        # Kernel of the observations
        k11 = kernel(X, X, **hyperparams)
        # Kernel of observations vs to-predict
        k12 = kernel(X, X_hat, **hyperparams)

        C_inv = torch.inverse(k11 + noise**2 * torch.eye(k11.shape[0]))

        kT_Cinv = torch.matmul(k12.T, C_inv)

        # Compute posterior mean
        mu = torch.matmul(kT_Cinv, t)

        # Compute the posterior covariance
        C = kernel(X_hat, X_hat, **hyperparams)
        cov = C - torch.matmul(kT_Cinv, k12)
        # ---------------------- student exercise --------------------------------- #

    return mu, cov

## Performing Gaussian process regression
Now we can perform regression using our GP model. We need to set the hyperparameters, let's assume we know them for now.

In [ ]:
hyperparams["noise"] = 0.2
hyperparams["sig_f"] = 1
hyperparams["length"] = 1

In [ ]:
# Compute the posterior mean and covariance
n1 = 8  # Number of points to condition on (observation locations)
n2 = 250  # Number of points in posterior (prediction locations)
n_sample_post = 12  # Number of functions that will be sampled from the posterior
domain_data = [-4, 4]
domain = [-10, 10]

# Sample observations (X1, t) on the function
torch.manual_seed(seed)
X1 = torch.rand((n1, 1)) * (domain_data[1] - domain_data[0]) + domain_data[0]

rng = torch.manual_seed(1)
t = f_data(X1, rng)

# Predict points at uniform spacing to capture function
X2 = torch.linspace(domain[0], domain[1], n2).view(-1, 1)

# Compute posterior mean and covariance
mu_pred, cov_pred = GP(X1, t, X2, cov_func, hyperparams)

# Compute the standard deviation at the test points to be plotted
sig = torch.sqrt(torch.diag(cov_pred))

# Draw some samples of the posterior
y2 = torch.tensor(
    np.random.multivariate_normal(
        mean=mu_pred.view(-1),
        cov=cov_pred,
        size=n_sample_post,
    )
)

Given these assumed hyperparameters, we can plot our predictions and the posterior distribution.

In [ ]:
# Plot the posterior distribution and some samples
def plot_preds(mu_pred, X1, X2, y2, t):
    mu_pred = mu_pred.flatten()
    X1 = X1.flatten()
    X2 = X2.flatten()
    t = t.flatten()

    fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, figsize=(8, 6))

    # Plot the distribution of the function (mean, covariance)
    ax1.plot(X2, f_truth(X2), "k--", label="Ground truth")
    ax1.fill_between(
        X2.flatten(),
        mu_pred - 1.96 * sig,
        mu_pred + 1.96 * sig,
        color="C0",
        alpha=0.15,
        label="95% conf. interval",
    )
    ax1.plot(X2, mu_pred, "-", color="C0", lw=2, label="Mean")
    ax1.plot(X1, t, "ko", linewidth=2, label="Observations")
    ax1.set_xlabel("$x$", fontsize=13)
    ax1.set_ylabel("$y$", fontsize=13)
    ax1.set_title("Posterior distribution and observation data")
    ax1.axis([domain[0], domain[1], -3, 3])
    ax1.legend(loc="upper left")

    # Plot some samples from this function
    for y_samp in y2:
        ax2.plot(X2, y_samp, "-")

    ax2.set_xlabel("$x$", fontsize=13)
    ax2.set_ylabel("$y$", fontsize=13)
    ax2.set_title(f"{n_sample_post} different function realizations from the posterior")
    ax2.axis([domain[0], domain[1], -3, 3])
    ax2.set_xlim([domain[0], domain[1]])
    plt.tight_layout()
    plt.show()


plot_preds(mu_pred, X1, X2, y2, t)

We have fit a GP to our data!
The top figure is the most interesting one, spend some time thinking about whether the model predictions make sense.
- Do the mean and variance reflect the observations accurately?
- What happens to the predictions of the GP far away from the data points?
- What do you expect to happen to happen here for the other kernel you implemented?
- Try the other more complex `f_truth` function and observe what happens.

## Model selection
Now we turn to model selection. We can infer the hyperparameters $\sigma_f$, $l$, and the noise directly from the data, without the need for a validation set.
We will find the optimal choice for these parameters by minimizing the log marginal likelihood.

### Loss function
Recall from the lecture that the log marginal likelihood reads

$$
\ln p(\mathbf{t}\vert\sigma_f,\ell,\beta) =
-\frac{1}{2}\ln\vert\mathbf{K}+\beta^{-1}\mathbf{I}\vert
-\frac{1}{2}\mathbf{t}^T\left(\mathbf{K}+\beta^{-1}\mathbf{I}\right)^{-1}\mathbf{t}
-\frac{N}{2}\ln\left(2\pi\right).
$$

The function below implements this expression in two different ways, the first one being more computionally robust and the second being the exact expression above.

In [ ]:
# Gaussian process log marginal likelihood
def GP_logmarglike(X, t, kernel, hyperparams):
    """
    Calculate the log marginal likelihood based on the observations (X, t) for a given kernel
    """
    # Kernel of the observations
    k11 = kernel(X, X, **hyperparams)

    C = k11 + hyperparams["noise"] ** 2 * torch.eye(k11.shape[0])

    # computationally efficient and numerically robust way:
    L = torch.linalg.cholesky(C)
    logmarglike = (
        - 0.5 * torch.sum(torch.log(torch.diagonal(L)))
        - 0.5 * torch.linalg.norm(torch.linalg.solve_triangular(L, t, upper=False)) **2
        - 0.5 * len(t) * torch.log(torch.tensor(2 * torch.pi))
    )

    # direct implementation of the expression above (also works but might be prone to numerical noise)
    # logmarglike = (
    #     - 0.5 * torch.log(torch.linalg.det(C))
    #     - 0.5 * torch.matmul(t.T,torch.matmul(torch.linalg.inv(C),t))
    #     - 0.5 * len(t) * torch.log(torch.tensor(2 * torch.pi))
    # )

    return logmarglike

We can now optimize the hyperparameters. After selecting initial values, we perform a gradient-descent based optimization.
This code is created for the squared exponential kernel, but it could be adapted for other kernels with relative ease.

After setting some initial values, we log-scale them for robustness and to avoid negative values. Then we can implement an optimization loop.

**NOTE**: If you want to run the optimization loop again, also rerun this block!

In [ ]:
# Initial values
sig_init = 1.0
noise_init = 1.0
length_init = 1.0

# Initialize variables and do not forget to track gradients
logsig = torch.log(torch.tensor([sig_init])).requires_grad_(True)
lognoise = torch.log(torch.tensor([noise_init])).requires_grad_(True)
loglen = torch.log(torch.tensor([length_init])).requires_grad_(True)

### Exercise: Empirical Bayes with Adam

In the exercise on empirical Bayes for basis function models we used BFGS to find an optimum. Here we demonstrate that you can also use Adam for that, since we are used to use that for neural nets.

Use the block below to implement an Adam training loop. You can use previous implementations as inspiration, but **do not forget to `torch.exp` your hyperparameters before using them!**

In [ ]:
# Create an Adam optimizer object
adam = torch.optim.Adam((logsig, lognoise, loglen), lr=0.01)

loglikes = []

# ---------------------- student exercise --------------------------------- #
n_epochs = 1000

for i in range(n_epochs):
    adam.zero_grad()

    hyperparams["sig_f"] = torch.exp(logsig)
    hyperparams["noise"] = torch.exp(lognoise)
    hyperparams["length"] = torch.exp(loglen)
    loss = -GP_logmarglike(X1, t, cov_func, hyperparams)

    loglikes.append(-loss.item())

    loss.backward()
    adam.step()
# ---------------------- student exercise --------------------------------- #

# Plot how the marginal likelihood evolves with epochs

# ---------------------- student exercise --------------------------------- #
plt.plot(loglikes)
plt.xlabel("Epoch")
plt.ylabel("Log marginal likelihood")
plt.show()
# ---------------------- student exercise --------------------------------- #

print("Optimum hyperparameters:", hyperparams)

The initial values of the hyperparameters can play a big role, the optimization algorithm could get trapped in a local maximum.
It is, therefore, best practice to try a couple different initializations.
Here we plot what the result looks like for the estimated hyperparameters:

In [ ]:
# Compute results
mu_pred, cov_pred = GP(X1, t, X2, cov_func, hyperparams)

# Compute the standard deviation at the test points to be plotted
sig = torch.sqrt(torch.diag(cov_pred))

# Draw some samples of the posterior
y2 = torch.tensor(
    np.random.multivariate_normal(
        mean=mu_pred.view(-1),
        cov=cov_pred.detach(),
        size=n_sample_post,
    )
)

# Plot
plot_preds(mu_pred, X1, X2, y2, t)

## Review
Guassian Processes regression is a useful non-parametric method.
You've seen that we need to invert a matrix of size N x N, where N is the number of data points, to obtain the posterior distribution.
This can become prohibitively expensive when dealing with large datasets.
Hence, for some problems parametric models, e.g. neural networks, are preferred because their training and prediction cost scale better with dataset size.
Still, GPs are a powerful tool — especially in the low data regime — and their inherent probabilistic nature makes them suitable for many engineering applications.

At this point you should have developed a good understanding of the concepts behind Gaussian process regression and the central role the kernel function takes therein.